# RACE Project Runner (Colab + VSCode)

This notebook is the **single place** to run setup, preprocessing, Model A training, and quick checks.

If you are using Colab extension in VSCode, run cells top-to-bottom.

## What this notebook does
- Verifies dataset files exist in `data/raw/`
- Runs preprocessing (`src.preprocessing`)
- Trains Model A (`src.model_a_train`)
- Shows saved metrics and generated artifacts

## Expected dataset files
- `data/raw/train.csv`
- `data/raw/dev.csv` (or `data/raw/val.csv`)
- `data/raw/test.csv`

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

ROOT = Path.cwd()
print('Current working directory:', ROOT)
print('Python:', sys.version)

# If notebook starts inside notebooks/, move to project root.
if ROOT.name == 'notebooks':
    %cd ..
    ROOT = Path.cwd()

print('Project root:', ROOT)

In [ ]:
# Install dependencies (safe to re-run)
!pip -q install -r requirements.txt

In [ ]:
# Check dataset files
from pathlib import Path

raw_dir = Path('data/raw')
expected = ['train.csv', 'test.csv']
optional_val = ['dev.csv', 'val.csv']

for f in expected:
    print(f, '->', (raw_dir / f).exists())
print('dev/val exists ->', any((raw_dir / f).exists() for f in optional_val))

if not (raw_dir / 'train.csv').exists() or not (raw_dir / 'test.csv').exists() or not any((raw_dir / f).exists() for f in optional_val):
    raise FileNotFoundError('Missing required CSVs in data/raw. Add train/test + dev or val first.')

In [ ]:
# Run preprocessing
!python -m src.preprocessing --raw-dir data/raw --processed-dir data/processed

In [ ]:
# Inspect preprocessing outputs
manifest_path = Path('data/processed/manifest.json')
if not manifest_path.exists():
    raise FileNotFoundError('manifest.json not found. Preprocessing may have failed.')

manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
print(json.dumps({
    'mcq_splits': manifest.get('mcq_splits'),
    'verify_splits': manifest.get('splits'),
    'ohe_feature_dim': manifest.get('ohe_feature_dim'),
    'tfidf_feature_dim': manifest.get('tfidf_feature_dim')
}, indent=2))

In [ ]:
# Train Model A (full)
!python -m src.model_a_train --processed-dir data/processed --output-dir models/model_a/traditional

In [ ]:
# Faster debug run (optional). Uncomment to use.
# !python -m src.model_a_train --processed-dir data/processed --output-dir models/model_a/traditional_debug --max-train-rows 20000 --generation-max-train-mcq 5000

In [ ]:
# Read Model A metrics
metrics_path = Path('models/model_a/traditional/metrics_summary.json')
if not metrics_path.exists():
    raise FileNotFoundError('Model A metrics file not found. Train step may have failed.')

metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
print('Config:')
print(json.dumps(metrics['config'], indent=2))
print('\nAvailable sections:', list(metrics.keys()))
print('\nVerification models:', list(metrics['results'].keys()))
if 'generation' in metrics:
    print('\nGeneration summary:')
    print(json.dumps(metrics['generation'], indent=2))

In [ ]:
# Quick artifact check
artifact_dir = Path('models/model_a/traditional')
for p in sorted(artifact_dir.glob('*')):
    print(p.name)